# Tilt census statistics

This notebook produces the numerical results used in the paper. Calculation and formatting functions live in `tilt_stats_tools.py`; the notebook only loads data, runs each report and displays its outputs.

Daily statistics describe **eddy-days**. Per-eddy statistics are reported separately. Direction statistics omit very small tilts because their bearing is poorly defined.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Markdown

# Run from either this folder, the tilt-analysis folder or the repository root.
candidates = [Path.cwd(), Path.cwd() / 'census',
              Path.cwd() / 'seacofs_eddy_tilt_analysis' / 'census',
              Path.cwd().parent / 'census']
CENSUS_DIR = next(p for p in candidates if (p / 'tilt_stats_tools.py').exists())
sys.path[:0] = [str(CENSUS_DIR), str(CENSUS_DIR.parent)]

import seacofs_tilt_tools as tilt
import tilt_stats_tools as stats

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df_eddies, df_tilt = tilt.load_tilt_tables(paths, add_regions=True, grid=grid)
df_eddies = df_eddies.sort_values(['Cyc', 'Eddy', 'Day']).copy()
print(f'{len(df_eddies):,} eddy-days; {df_eddies.groupby(["Cyc", "Eddy"]).ngroups:,} eddies loaded.')

## 1. Tilt-distance census

Coverage is calculated against Delta-method-eligible days after excluding the first and last three observations of each track.

In [ ]:
tilt_report = stats.tilt_census_report(df_eddies, first_last_days=3)

display(Markdown('### Dataset and tilt-estimate coverage'))
display(tilt_report['census'].round(2))
display(Markdown('### Daily tilt distance (eddy-days)'))
display(tilt_report['daily'].round(2))
display(Markdown('### Per-eddy tilt-distance metrics'))
display(tilt_report['per_eddy'].round(2))
display(Markdown('### Broad sectors: upstream + S1 versus downstream + S2'))
display(tilt_report['sector'].round(2))
display(Markdown('### Individual regions'))
display(tilt_report['regional'].round(2))

In [ ]:
print('COPYABLE TILT-DISTANCE RESULTS\n')
print('\n'.join(stats.tilt_census_sentences(tilt_report)))

## 2. Tilt-direction census

The first table uses the full population. The second is the requested **non-shelf census**, pooling only U1, U2, D1 and D2. The third compares offshore upstream (U1/U2) with offshore downstream (D1/D2); S1 and S2 are excluded from both.

In [ ]:
MIN_TILT_KM = 5.0
direction_report = stats.direction_census_report(df_eddies, minimum_tilt_km=MIN_TILT_KM)

display(Markdown('### All regions'))
display(direction_report['overall'].round(2))
display(Markdown('### Non-shelf only: U1, U2, D1 and D2 pooled'))
display(direction_report['offshore'].round(2))
display(Markdown('### Non-shelf upstream versus downstream'))
display(direction_report['offshore_sector'].round(2))
display(Markdown('### Each non-shelf region'))
display(direction_report['offshore_regional'].round(2))

In [ ]:
print('COPYABLE TILT-DIRECTION RESULTS\n')
print('\n'.join(stats.direction_census_sentences(direction_report)))

print('\nNON-SHELF UPSTREAM/DOWNSTREAM DETAIL\n')
for (cyc, sector), row in direction_report['offshore_sector'].iterrows():
    print(f'{cyc} {sector.lower()}: median tilt = {row.median_tilt_km:.1f} km; '
          f'tilt >40 km = {row.over_40_km_pct:.1f}%; mean bearing = '
          f'{row.mean_direction_deg:.1f}°; R = {row.directional_concentration_R:.2f}.')

## 3. Alignment with the PV gradient (optional)

This runs when `PV_grad_theta` is already present. It evaluates the stated hypothesis: CE tilt is aligned with the PV gradient, whereas AE tilt is aligned with the opposite direction. Results are shown for both the total population and non-shelf subset.

In [ ]:
if 'PV_grad_theta' not in direction_report['data'].columns:
    print('PV_grad_theta is not present; calculate PV-gradient terms before running this section.')
else:
    pv_report = stats.pv_alignment_report(direction_report['data'])
    display(Markdown('### All-region PV-gradient alignment'))
    display(pv_report['overall'].round(2))
    display(Markdown('### Non-shelf PV-gradient alignment'))
    display(pv_report['offshore'].round(2))
    display(Markdown('### Individual non-shelf regions'))
    display(pv_report['offshore_regional'].round(2))

## Interpretation notes

- `n` is the number of eddy-days; `n_eddies` is the number of distinct eddies represented.
- Bearings use 0° = north, 90° = east, 180° = south and 270° = west.
- `directional_concentration_R` ranges from 0 (directions broadly dispersed) to 1 (all bearings identical).
- For AEs, the preferred meridional/quadrant columns mean equatorward/northeastward. For CEs, they mean poleward/southeastward.
- These percentages describe observed eddy-days. Formal inference should account for repeated days within each eddy.